# Week 9 Lab: Preparing Data

Four short exercises, one per preparation step from this week's lectures: **outliers, scaling, missing values, and encoding**. Each one gives you data and asks you to try a technique and look at the effect.

**Core**: Exercises 1–4 (about 40 minutes). **Extension**: Exercise 5, an end-to-end analysis, if you finish early.

Run the setup cell first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#### **Exercise 1**

**Outliers.** `data/sample_data_with_outliers.csv` has customers' `Age` and `Income` (plus a `Purchase` label). A few values are clearly wrong.

A common rule (the **IQR rule**) calls a value an outlier if it lies more than 1.5 × IQR below the first quartile or above the third quartile, where IQR = Q3 − Q1:

$$\text{keep } x \text{ if } \; Q1 - 1.5 \cdot IQR \;\le\; x \;\le\; Q3 + 1.5 \cdot IQR$$

1. Complete `remove_outliers_iqr` (use `df[col].quantile(0.25)` and `.quantile(0.75)`) and run the cell. How many rows does it remove?
2. Now look at the data yourself in the next cell. Then write a **common-sense** filter that keeps only plausible rows: ages no greater than 110, and yearly incomes of at least $1,000.

In [ ]:
df = pd.read_csv('data/sample_data_with_outliers.csv')

def remove_outliers_iqr(df, columns):
    """Return a copy of df without rows that are IQR-rule outliers in any of `columns`."""
    df_clean = df.copy()
    for col in columns:
        # YOUR CODE HERE: compute Q1, Q3, IQR, the lower/upper bounds, and filter df_clean
        pass
    return df_clean

df_iqr = remove_outliers_iqr(df, ['Age', 'Income'])
print(f"IQR rule: {len(df)} rows before, {len(df_iqr)} after")

In [ ]:
# GIVEN: look at the extremes
print("Largest ages:    ", sorted(df['Age'])[-12:])
print("Smallest incomes:", sorted(df['Income'])[:22])

# YOUR CODE HERE: keep only plausible rows (Age <= 110 and Income >= 1000)
df_clean = df  # replace this line
print(f"Common-sense filter: {len(df)} rows before, {len(df_clean)} after")

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, col in enumerate(['Age', 'Income']):
    axes[2 * i].boxplot(df[col]);           axes[2 * i].set_title(f'{col}: before')
    axes[2 * i + 1].boxplot(df_clean[col]); axes[2 * i + 1].set_title(f'{col}: after')
plt.tight_layout()
plt.show()

**Question:** Why did the IQR rule miss values that are obviously wrong (a 145-year-old; a yearly income of $5)? What does that tell you about relying on a statistical rule alone?

_Answer here_

#### **Exercise 2**

**Scaling vs. transforming.** The data below has one exponentially distributed feature (`X1`, very skewed) and one normal feature (`X2`). The baseline model is trained on the raw features.

1. Run the cell to see the baseline accuracy.
2. Complete the two blocks: (a) scale **both** features with `StandardScaler` (fit on the training data only); (b) replace `X1` with `np.log1p(X1)` and leave `X2` alone.

In [ ]:
np.random.seed(42)
n_samples = 1000
X1 = np.random.exponential(scale=5, size=n_samples)   # skewed
X2 = np.random.normal(loc=50, scale=10, size=n_samples)
y = np.array([1 if x1 + 0.001 * x2 > 1 else 0 for x1, x2 in zip(X1, X2)])
flip = np.random.choice(n_samples, size=int(0.1 * n_samples), replace=False)
y[flip] = 1 - y[flip]                                  # 10% label noise

X = np.column_stack((X1, X2))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print(f"Majority-class rate:  {max(y_test.mean(), 1 - y_test.mean()):.3f}")
print(f"Raw features:         {baseline.score(X_test, y_test):.3f}")

# (a) StandardScaler on both features
# YOUR CODE HERE

# (b) log-transform X1 (column 0) only
# YOUR CODE HERE

**Question:** One of the two changes helps a lot and the other doesn't change the accuracy at all. Which is which, and why? (Hint: a linear model can multiply a feature by any weight it likes. Can it "undo" a rescaling? Can it undo a log?)

_Answer here_

#### **Exercise 3**

**Missing values.** About 20% of Titanic passengers have no recorded `age`. The setup code loads the data and defines `evaluate(X_imputed)`, which reports 5-fold cross-validated accuracy of a logistic regression.

Fill in each version of `X` using a different way of handling the missing ages, then evaluate it:
1. Forward fill: `X['age'].ffill()` (copy the previous row's age), then `.bfill()` for any gap at the very top
2. `SimpleImputer(strategy='mean')`
3. `SimpleImputer(strategy='most_frequent')`
4. `KNNImputer(n_neighbors=5)` on all columns (it uses the other features to guess the age)

Remember: `SimpleImputer` needs 2-D input, so pass `X[['age']]`, not `X['age']`.

In [ ]:
# The file is sorted by passenger class, so shuffle once before cross-validating
titanic = pd.read_csv('data/titanic.csv').sample(frac=1, random_state=42)
X = titanic[['pclass', 'sex', 'age', 'sibsp', 'parch']].copy()
X['sex'] = (X['sex'] == 'female').astype(int)
y = titanic['survived']
print(f"Missing ages: {X['age'].isna().sum()} of {len(X)}")

def evaluate(X_imputed, label):
    acc = cross_val_score(LogisticRegression(max_iter=1000), X_imputed, y, cv=5).mean()
    print(f"{label:15} accuracy = {acc:.3f}")

X_ffill = X.copy()
X_mean = X.copy()
X_freq = X.copy()
# YOUR CODE HERE: fill in the age column of each copy, build X_knn, then call evaluate() four times

**Questions**

1. How much do the four methods differ? What does that suggest about how much the choice matters *for this dataset and model*?
2. Forward fill copies the previous passenger's age. Why is that a strange choice for this data, even if the score looks fine? When would forward fill make sense?

_Answer here_

#### **Exercise 4**

**Encoding.** One-hot encode **both** text columns of `sample` below (`color` and `target`), keeping `yumminess` as it is. Use a `ColumnTransformer` with a `OneHotEncoder(sparse_output=False)` and `remainder='passthrough'`, and show the result as a DataFrame with `get_feature_names_out()` as the column names.

In [ ]:
sample = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'blue', 'green'],
    'target': ['apple', 'sky', 'grass', 'sky', 'moon', 'apple'],
    'yumminess': [10, 2, 3, 1, 6, 9]
})

# YOUR CODE HERE

**Question:** Why shouldn't you one-hot encode `yumminess`? And why would it be a mistake to encode `color` as red=0, blue=1, green=2 instead?

_Answer here_

---

### Extension exercise (optional)

#### **Exercise 5** (extension)

**End to end.** `data/synthetic_patient_data_unbalanced.csv` simulates patients examined for cardiovascular disease (`has_disease`: 1 = yes). It has numeric columns (`age`, `bmi`, `systolic_bp`, `diastolic_bp`, `cholesterol_level`), categorical ones (`gender`, `exercise_frequency`, `smoker`, `family_history`, `diet_quality`, `us_state`), some missing values, and two columns you should think about before using: `risk` and `shoe_size`.

1. Build a `Pipeline` with a `ColumnTransformer` (impute + scale the numeric columns; impute + one-hot encode the categorical ones) and a `LogisticRegression`.
2. Report accuracy, precision, recall, and F1 with 5-fold `cross_validate`.
3. The classes are very unbalanced: about 10 of every 11 patients **have** the disease. What accuracy would you get by always predicting "disease"? Which of your metrics is misleading, and why?

**Bonus:** add SMOTE to rebalance the classes (see the optional reading `lectures/09-data-preparation/extra/7-imbalanced-classes.ipynb`). You'll need `imblearn`'s `Pipeline` instead of sklearn's. How do the metrics change?

In [ ]:
from sklearn.model_selection import cross_validate
patients = pd.read_csv('data/synthetic_patient_data_unbalanced.csv', index_col=0)
print(patients['has_disease'].value_counts())
print(f"Always predicting 'disease' would score {patients['has_disease'].mean():.3f} accuracy")

# YOUR CODE HERE

_Answer here_